# DATA CLEANING

In [1]:
import pandas as pd

# Define your tables
table_names = [
    'orders',
    'customers', 
    'products',
    'sellers',
    'order_payments',
    'order_items',
    'order_reviews',
    'geoloc'
]

# Load all into a dictionary
datasets = {
    name: pd.read_parquet(f'../data/outputs/{name}_raw.parquet')
    for name in table_names
}

### Validity - Assigning relevant data types

In [2]:
# Orders 

datasets['orders']['order_status'] = datasets['orders']['order_status'].astype('category')
datasets['orders']['order_purchase_timestamp'] = pd.to_datetime(datasets['orders']['order_purchase_timestamp'])
datasets['orders']['order_approved_at'] = pd.to_datetime(datasets['orders']['order_approved_at'])
datasets['orders']['order_delivered_carrier_date'] = pd.to_datetime(datasets['orders']['order_delivered_carrier_date'])
datasets['orders']['order_delivered_customer_date'] = pd.to_datetime(datasets['orders']['order_delivered_customer_date'])
datasets['orders']['order_estimated_delivery_date'] = pd.to_datetime(datasets['orders']['order_estimated_delivery_date'])

# Customers 

datasets['customers']['customer_city'] = datasets['customers']['customer_city'].astype('category')
datasets['customers']['customer_state'] = datasets['customers']['customer_state'].astype('category')

# Products
datasets['products']['product_photos_qty'] = datasets['products']['product_photos_qty'].fillna(0).astype('Int64')
# Sellers
datasets['sellers']['seller_city'] = datasets['sellers']['seller_city'].astype('category')
datasets['sellers']['seller_state'] = datasets['sellers']['seller_state'].astype('category')

# Order payments
datasets['order_payments']['payment_type'] = datasets['order_payments']['payment_type'].astype('category')

# Order items
datasets['order_items']['order_item_id'] = datasets['order_items']['order_item_id'].astype('str')
datasets['order_items']['shipping_limit_date'] = pd.to_datetime(datasets['order_items']['shipping_limit_date'])

# Order reviews
datasets['order_reviews']['review_creation_date'] = pd.to_datetime(datasets['order_reviews']['review_creation_date'])
datasets['order_reviews']['review_answer_timestamp'] = pd.to_datetime(datasets['order_reviews']['review_answer_timestamp'])

# Geo-location
datasets['geoloc']['geolocation_city'] = datasets['geoloc']['geolocation_city'].astype('category')
datasets['geoloc']['geolocation_state'] = datasets['geoloc']['geolocation_state'].astype('category')

### Completeness - dealing with missing value

In [3]:
datasets['orders'].isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [4]:
## Orders
orders = datasets['orders']

In [5]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Cannot impute the date columns. Hence, need to exclude the missing data and flag it while doing time related analysis.

In [6]:
## Products
datasets['products'].isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty              0
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [7]:
# Order reviews
datasets['order_reviews'].isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

### Accuracy Check

In [9]:
# Orders

# checking if the order status has any noise data
print(orders.order_status.unique())

# check to ensure order purchase date is older than order delivered date - none
orders[orders['order_purchase_timestamp'] > orders['order_delivered_customer_date']]

# Products
products = datasets['products']
# replacing '_' with spaces
products.product_category_name = products.product_category_name.str.lower().str.replace('_',' ')

['delivered', 'invoiced', 'shipped', 'processing', 'unavailable', 'canceled', 'created', 'approved']
Categories (8, object): ['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']


In [ ]:
datasets.keys()

dict_keys(['orders', 'customers', 'products', 'sellers', 'order_payments', 'order_items', 'order_reviews', 'geoloc'])

In [ ]:
# check for any orphan order ids
orderids = set(orders.order_id)
orderids2 = set(datasets['order_items'].order_id)
orderids2 - orderids2

set()

In [34]:
# check for orphan customer ids
customerid = set(datasets['customers'].customer_id)
customerid2 = set(datasets['orders'].customer_id)
customerid - customerid2

set()

In [36]:
# check for orphan customer ids
productid = set(datasets['products'].product_id)
productid2 = set(datasets['order_items'].product_id)
productid - productid2

set()

In [42]:
#### Storing the processed data as an object
def store_raw_data_obj(dataframes: dict):
    for name, df in dataframes.items():
        df.to_parquet(f'../data/processed/{name}_processed.parquet', index = False)

store_raw_data_obj(datasets)